
# Journal-version IBM Geometry Validation for Midpoint-Projected Manifold A2G-QFL

This notebook converts the Python script into an interactive `.ipynb` workflow.

It generates a **journal-version geometry validation table** comparing:

- QoS-weighted linear aggregation,
- QoS-weighted circular/manifold aggregation,
- midpoint-projected manifold aggregation,
- adaptive midpoint-projected manifold aggregation,
- ideal simulator expectation values,
- optional IBM Quantum hardware expectation values using `EstimatorV2`.

The toy circuit is:

\[
|0\rangle \xrightarrow{R_y(\theta)} \text{measure } \langle Z\rangle,
\]

where ideally:

\[
\langle Z\rangle = \cos(\theta).
\]



## 0. Environment notes

Before opening Jupyter, activate your conda environment in Anaconda Prompt or PowerShell:

```powershell
conda activate ResearchAssistDeakin
jupyter notebook
```

If IBM Runtime is not installed in the environment, run the optional installation cell below.


In [ ]:

# Optional: install IBM Runtime package if missing.
# Run this cell only if `import qiskit_ibm_runtime` fails.

# !pip install qiskit qiskit-ibm-runtime pandas numpy


In [1]:

from __future__ import annotations

import os
import getpass
from dataclasses import dataclass
from typing import List, Dict, Any

import numpy as np
import pandas as pd



## 1. Manifold helpers for one-dimensional torus / angle geometry

For one quantum rotation angle, the natural geometry is a circle \(S^1\).  
For many rotation angles, the geometry becomes a product of circles, i.e., a torus \(\mathbb{T}^d\).

In this toy validation, we use degree values for readability.


In [2]:

def wrap_deg(x: float | np.ndarray) -> float | np.ndarray:
    """Wrap degrees to [-180, 180)."""
    return ((np.asarray(x) + 180.0) % 360.0) - 180.0


def weighted_linear_mean_deg(angles_deg: List[float], weights: List[float]) -> float:
    """Naive weighted linear mean in degree coordinates."""
    a = np.asarray(angles_deg, dtype=float)
    w = np.asarray(weights, dtype=float)
    w = w / (w.sum() + 1e-12)
    return float(np.sum(w * a))


def weighted_circular_mean_deg(
    angles_deg: List[float],
    weights: List[float],
    eps: float = 1e-10,
) -> float:
    """QoS-weighted circular mean on S^1 with ambiguity check."""
    a = np.deg2rad(np.asarray(angles_deg, dtype=float))
    w = np.asarray(weights, dtype=float)
    w = w / (w.sum() + 1e-12)

    s = np.sum(w * np.sin(a))
    c = np.sum(w * np.cos(a))
    resultant = np.sqrt(s**2 + c**2)

    if resultant < eps:
        # Circular mean is not unique. Returning 0 is deterministic,
        # but this case should be flagged in the table/paper.
        return 0.0

    return float(wrap_deg(np.rad2deg(np.arctan2(s, c))))


def torus_log_deg(base_deg: float, point_deg: float) -> float:
    """
    Log map on S^1 in degree representation:
    shortest wrapped displacement from base_deg to point_deg.
    """
    return float(wrap_deg(point_deg - base_deg))


def torus_exp_deg(base_deg: float, tangent_deg: float) -> float:
    """
    Exp map on S^1 in degree representation:
    move from base_deg by tangent_deg and wrap.
    """
    return float(wrap_deg(base_deg + tangent_deg))


def manifold_dispersion_rad2(theta_t_deg: float, angles_deg: List[float], weights: List[float]) -> float:
    """Weighted squared wrapped distance from theta_t to local client angles."""
    w = np.asarray(weights, dtype=float)
    w = w / (w.sum() + 1e-12)
    diffs_rad = np.deg2rad([torus_log_deg(theta_t_deg, a) for a in angles_deg])
    return float(np.sum(w * diffs_rad**2))


def adaptive_beta(beta0: float, dispersion: float, beta_lambda: float, beta_min: float) -> float:
    """Adaptive geometry gain beta_t = max(beta_min, beta0/(1+lambda D_t))."""
    return float(max(beta_min, beta0 / (1.0 + beta_lambda * dispersion)))


def midpoint_projected_update_deg(
    theta_t_deg: float,
    manifold_target_deg: float,
    beta: float = 1.0,
) -> Dict[str, float]:
    """
    Target-based midpoint-projected update on S^1.

    This version is suitable for the one-qubit journal validation table.
    It shows the server motion from the previous global angle theta_t
    toward the QoS-weighted circular/manifold target.
    """
    v_deg = torus_log_deg(theta_t_deg, manifold_target_deg)

    raw_midpoint_deg = theta_t_deg + 0.5 * beta * v_deg
    midpoint_deg = wrap_deg(raw_midpoint_deg)

    raw_next_deg = theta_t_deg + beta * v_deg
    theta_next_deg = wrap_deg(raw_next_deg)

    projection_shift_deg = abs(raw_next_deg - theta_next_deg)

    return {
        "tangent_to_target_deg": float(v_deg),
        "raw_midpoint_deg": float(raw_midpoint_deg),
        "theta_midpoint_deg": float(midpoint_deg),
        "raw_next_deg": float(raw_next_deg),
        "theta_next_deg": float(theta_next_deg),
        "projection_shift_deg": float(projection_shift_deg),
    }

def z_expectation_ideal(theta_deg: float) -> float:
    """Ideal <Z> for Ry(theta)|0> is cos(theta)."""
    return float(np.cos(np.deg2rad(theta_deg)))



## 2. Experiment cases

Cases A-D mirror the earlier conference-style periodicity table, but this journal version adds:

- previous global angle \(\theta_t\),
- QoS/trust weights \(W_i\),
- midpoint-projected update,
- adaptive midpoint-projected update.

Case E is a QoS-skewed wrap-around case.


In [3]:

@dataclass
class GeometryCase:
    case: str
    theta_t_deg: float
    angles_deg: List[float]
    qos_weights: List[float]
    note: str


def default_cases() -> List[GeometryCase]:
    return [
        GeometryCase(
            case="A",
            theta_t_deg=0.0,
            angles_deg=[5.7, -11.5, 2.9],
            qos_weights=[1/3, 1/3, 1/3],
            note="non-wrap control case",
        ),
        GeometryCase(
            case="B",
            theta_t_deg=150.0,
            angles_deg=[174.3, -174.3],
            qos_weights=[0.5, 0.5],
            note="two-client wrap-around case",
        ),
        GeometryCase(
            case="C",
            theta_t_deg=0.0,
            angles_deg=[-85.9, 0.0, 85.9],
            qos_weights=[1/3, 1/3, 1/3],
            note="symmetric control case",
        ),
        GeometryCase(
            case="D",
            theta_t_deg=150.0,
            angles_deg=[168.5, -171.4, 177.1],
            qos_weights=[1/3, 1/3, 1/3],
            note="three-client wrap-around case",
        ),
        GeometryCase(
            case="E",
            theta_t_deg=100.0,
            angles_deg=[170.0, -175.0, 178.0],
            qos_weights=[0.15, 0.55, 0.30],
            note="QoS-skewed wrap-around case",
        ),
    ]



## 3. Build the journal validation table without IBM hardware

This produces the simulator-only table using the ideal relation:

\[
\langle Z\rangle = \cos(\theta).
\]


In [4]:

def build_table(
    beta0: float = 1.0,
    beta_lambda: float = 0.25,
    beta_min: float = 0.05,
) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []

    for c in default_cases():
        lin_deg = weighted_linear_mean_deg(c.angles_deg, c.qos_weights)
        circ_deg = weighted_circular_mean_deg(c.angles_deg, c.qos_weights)

        mp = midpoint_projected_update_deg(
            theta_t_deg=c.theta_t_deg,
            manifold_target_deg=circ_deg,
            beta=beta0,
        )

        D = manifold_dispersion_rad2(c.theta_t_deg, c.angles_deg, c.qos_weights)
        beta_eff = adaptive_beta(beta0, D, beta_lambda, beta_min)

        amp = midpoint_projected_update_deg(
            theta_t_deg=c.theta_t_deg,
            manifold_target_deg=circ_deg,
            beta=beta_eff,
        )

        row = {
            "case": c.case,
            "K": len(c.angles_deg),
            "note": c.note,
            "theta_t_deg": c.theta_t_deg,
            "client_angles_deg": ", ".join(f"{x:.1f}" for x in c.angles_deg),
            "qos_weights": ", ".join(f"{x:.2f}" for x in c.qos_weights),
            "theta_lin_deg": lin_deg,
            "theta_circ_target_deg": circ_deg,
            "theta_midpoint_deg": mp["theta_midpoint_deg"],
            "theta_mp_deg": mp["theta_next_deg"],
            "manifold_dispersion_D_rad2": D,
            "beta_eff": beta_eff,
            "theta_adaptive_midpoint_deg": amp["theta_midpoint_deg"],
            "theta_adaptive_mp_deg": amp["theta_next_deg"],
            "Z_sim_lin": z_expectation_ideal(lin_deg),
            "Z_sim_circ_target": z_expectation_ideal(circ_deg),
            "Z_sim_mp": z_expectation_ideal(mp["theta_next_deg"]),
            "Z_sim_adaptive_mp": z_expectation_ideal(amp["theta_next_deg"]),
        }
        rows.append(row)

    df = pd.DataFrame(rows)
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    df[numeric_cols] = df[numeric_cols].round(6)
    return df


In [5]:

# Parameters for journal midpoint geometry validation.
beta0 = 1.0
beta_lambda = 0.25
beta_min = 0.05

df = build_table(beta0=beta0, beta_lambda=beta_lambda, beta_min=beta_min)
df


,case,K,note,theta_t_deg,client_angles_deg,qos_weights,theta_lin_deg,theta_circ_target_deg,theta_midpoint_deg,theta_mp_deg,manifold_dispersion_D_rad2,beta_eff,theta_adaptive_midpoint_deg,theta_adaptive_mp_deg,Z_sim_lin,Z_sim_circ_target,Z_sim_mp,Z_sim_adaptive_mp
0,A,3,non-wrap control case,0.0,"5.7, -11.5, 2.9","0.33, 0.33, 0.33",-0.966667,-0.952791,-0.476396,-0.952791,0.017582,0.995624,-0.474311,-0.948622,0.999858,0.999862,0.999862,0.999863
1,B,2,two-client wrap-around case,150.0,"174.3, -174.3","0.50, 0.50",0.000000,-180.000000,165.000000,-180.000000,0.284053,0.933695,164.005430,178.010860,1.000000,-1.000000,-1.000000,-0.999397
2,C,3,symmetric control case,0.0,"-85.9, 0.0, 85.9","0.33, 0.33, 0.33",0.000000,0.000000,0.000000,0.000000,1.498476,0.727474,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000
3,D,3,three-client wrap-around case,150.0,"168.5, -171.4, 177.1","0.33, 0.33, 0.33",58.066667,178.061683,164.030841,178.061683,0.260612,0.938832,163.172605,176.345211,0.528932,-0.999428,-0.999428,-0.997966
4,E,3,QoS-skewed wrap-around case,100.0,"170.0, -175.0, 178.0","0.15, 0.55, 0.30",-17.350000,-179.342799,140.328601,-179.342799,1.990355,0.667740,126.929022,153.858044,0.954501,-0.999934,-0.999934,-0.897705



## 4. Save simulator table to CSV


In [6]:

OUT_CSV = "journal_geometry_validation_table_sim.csv"
df.to_csv(OUT_CSV, index=False)
print(f"Saved: {OUT_CSV}")


Saved: journal_geometry_validation_table_sim.csv



## 5. Optional IBM hardware execution using EstimatorV2

Run this section only when:

1. your IBM Quantum account is active,
2. `qiskit-ibm-runtime` is installed,
3. you have access to a real backend such as `ibm_fez`.

Token handling:
- The notebook first checks the environment variable `IBM_QUANTUM_TOKEN`.
- If it is not set, it asks through `getpass`, so the token is not printed in the notebook output.

In PowerShell, you can set the token before launching Jupyter:

```powershell
$env:IBM_QUANTUM_TOKEN="your_token_here"
jupyter notebook
```


In [7]:

def ensure_ibm_account():
    """
    Configure IBM Quantum account safely.

    Preferred:
        set IBM_QUANTUM_TOKEN in the environment.

    Fallback:
        enter token through getpass.
    """
    from qiskit_ibm_runtime import QiskitRuntimeService

    try:
        # Try loading an existing saved account first.
        service = QiskitRuntimeService()
        return service
    except Exception:
        pass

    token = os.getenv("IBM_QUANTUM_TOKEN")
    if not token:
        token = getpass.getpass("Enter IBM Quantum token: ")

    QiskitRuntimeService.save_account(
        channel="ibm_quantum",
        token=token,
        set_as_default=True,
        overwrite=True,
    )
    return QiskitRuntimeService()


def run_ibm_estimator_for_angles(theta_deg_values: List[float], backend_name: str | None = None):
    """
    Run IBM EstimatorV2 for Ry(theta)|0> and observable Z.
    Requires qiskit-ibm-runtime and IBM account access.
    """
    from qiskit.circuit import Parameter, QuantumCircuit
    from qiskit.quantum_info import SparsePauliOp
    from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
    from qiskit_ibm_runtime import EstimatorV2 as Estimator

    service = ensure_ibm_account()

    if backend_name:
        backend = service.backend(backend_name)
    else:
        backend = service.least_busy(operational=True, simulator=False, min_num_qubits=1)

    theta = Parameter("theta")
    qc = QuantumCircuit(1)
    qc.ry(theta, 0)

    observable = SparsePauliOp("Z")

    pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
    isa_circuit = pm.run(qc)
    isa_observable = observable.apply_layout(isa_circuit.layout)

    params = np.deg2rad(np.asarray(theta_deg_values, dtype=float)).reshape(-1, 1)

    estimator = Estimator(mode=backend)
    job = estimator.run([(isa_circuit, isa_observable, params)])

    print("IBM backend:", backend.name)
    print("IBM job id:", job.job_id())

    result = job.result()[0]
    evs = np.asarray(result.data.evs, dtype=float).reshape(-1)

    stds = None
    if hasattr(result.data, "stds"):
        stds = np.asarray(result.data.stds, dtype=float).reshape(-1)

    return backend.name, job.job_id(), evs, stds


def add_ibm_columns(df: pd.DataFrame, backend_name: str | None = None) -> pd.DataFrame:
    angle_columns = [
        "theta_lin_deg",
        "theta_circ_target_deg",
        "theta_mp_deg",
        "theta_adaptive_mp_deg",
    ]

    flat_angles = []
    keys = []

    for idx, row in df.iterrows():
        for col in angle_columns:
            flat_angles.append(float(row[col]))
            keys.append((idx, col))

    backend, job_id, evs, stds = run_ibm_estimator_for_angles(flat_angles, backend_name)

    out = df.copy()
    out["ibm_backend"] = backend
    out["ibm_job_id"] = job_id

    for n, (idx, col) in enumerate(keys):
        label = col.replace("theta_", "Z_ibm_").replace("_deg", "")
        out.loc[idx, label] = evs[n]
        if stds is not None:
            out.loc[idx, label.replace("Z_ibm_", "Z_ibm_std_")] = stds[n]

    numeric_cols = out.select_dtypes(include=[np.number]).columns
    out[numeric_cols] = out[numeric_cols].round(6)
    return out


In [9]:

# Set this to True only when you want to submit jobs to IBM hardware.
RUN_IBM = True

# Use a backend name if you want a specific backend, e.g., "ibm_fez".
# If None, the notebook selects the least-busy operational QPU with at least one qubit.
IBM_BACKEND = "ibm_fez"  # or None

if RUN_IBM:
    df_ibm = add_ibm_columns(df, backend_name=IBM_BACKEND)
    display(df_ibm)
else:
    print("IBM execution skipped. Set RUN_IBM=True to submit hardware jobs.")


IBM backend: ibm_fez
IBM job id: d85f4qftjchs73br85r0


,case,K,note,theta_t_deg,client_angles_deg,qos_weights,theta_lin_deg,theta_circ_target_deg,theta_midpoint_deg,theta_mp_deg,...,ibm_backend,ibm_job_id,Z_ibm_lin,Z_ibm_std_lin,Z_ibm_circ_target,Z_ibm_std_circ_target,Z_ibm_mp,Z_ibm_std_mp,Z_ibm_adaptive_mp,Z_ibm_std_adaptive_mp
0,A,3,non-wrap control case,0.0,"5.7, -11.5, 2.9","0.33, 0.33, 0.33",-0.966667,-0.952791,-0.476396,-0.952791,...,ibm_fez,d85f4qftjchs73br85r0,0.982492,0.006634,0.988157,0.005907,0.994336,0.006264,0.992276,0.005102
1,B,2,two-client wrap-around case,150.0,"174.3, -174.3","0.50, 0.50",0.000000,-180.000000,165.000000,-180.000000,...,ibm_fez,d85f4qftjchs73br85r0,0.983007,0.006786,-0.993821,0.004478,-0.991761,0.005678,-0.997940,0.005320
2,C,3,symmetric control case,0.0,"-85.9, 0.0, 85.9","0.33, 0.33, 0.33",0.000000,0.000000,0.000000,0.000000,...,ibm_fez,d85f4qftjchs73br85r0,0.974253,0.006942,0.993306,0.005077,0.988157,0.005907,0.985582,0.007124
3,D,3,three-client wrap-around case,150.0,"168.5, -171.4, 177.1","0.33, 0.33, 0.33",58.066667,178.061683,164.030841,178.061683,...,ibm_fez,d85f4qftjchs73br85r0,0.541195,0.013249,-0.991246,0.007002,-0.980947,0.007950,-0.981977,0.005355
4,E,3,QoS-skewed wrap-around case,100.0,"170.0, -175.0, 178.0","0.15, 0.55, 0.30",-17.350000,-179.342799,140.328601,-179.342799,...,ibm_fez,d85f4qftjchs73br85r0,0.940268,0.008513,-0.990731,0.004835,-0.989701,0.006017,-0.888260,0.009310


In [10]:

if "df_ibm" in globals():
    OUT_IBM_CSV = "journal_geometry_validation_table_ibm.csv"
    df_ibm.to_csv(OUT_IBM_CSV, index=False)
    print(f"Saved: {OUT_IBM_CSV}")
else:
    print("No IBM table available yet. Run the IBM section first.")


Saved: journal_geometry_validation_table_ibm.csv



## 6. Generate LaTeX table from the latest available DataFrame

This creates a compact LaTeX table. You can edit the column list below depending on journal page width.


In [11]:

report_df = df_ibm.copy() if "df_ibm" in globals() else df.copy()

latex_cols = [
    "case",
    "K",
    "theta_t_deg",
    "client_angles_deg",
    "qos_weights",
    "theta_lin_deg",
    "theta_circ_target_deg",
    "theta_midpoint_deg",
    "theta_mp_deg",
    "manifold_dispersion_D_rad2",
    "beta_eff",
    "theta_adaptive_mp_deg",
    "Z_sim_lin",
    "Z_sim_mp",
    "Z_sim_adaptive_mp",
]

# Add IBM columns if available.
for c in ["Z_ibm_lin", "Z_ibm_circ_target", "Z_ibm_mp", "Z_ibm_adaptive_mp"]:
    if c in report_df.columns and c not in latex_cols:
        latex_cols.append(c)

latex_table = report_df[latex_cols].to_latex(index=False, escape=False)
print(latex_table)

with open("journal_geometry_validation_table.tex", "w", encoding="utf-8") as f:
    f.write(latex_table)

print("Saved: journal_geometry_validation_table.tex")


\begin{tabular}{lrrllrrrrrrrrrrrrrr}
\toprule
case & K & theta_t_deg & client_angles_deg & qos_weights & theta_lin_deg & theta_circ_target_deg & theta_midpoint_deg & theta_mp_deg & manifold_dispersion_D_rad2 & beta_eff & theta_adaptive_mp_deg & Z_sim_lin & Z_sim_mp & Z_sim_adaptive_mp & Z_ibm_lin & Z_ibm_circ_target & Z_ibm_mp & Z_ibm_adaptive_mp \\
\midrule
A & 3 & 0.000000 & 5.7, -11.5, 2.9 & 0.33, 0.33, 0.33 & -0.966667 & -0.952791 & -0.476396 & -0.952791 & 0.017582 & 0.995624 & -0.948622 & 0.999858 & 0.999862 & 0.999863 & 0.982492 & 0.988157 & 0.994336 & 0.992276 \\
B & 2 & 150.000000 & 174.3, -174.3 & 0.50, 0.50 & 0.000000 & -180.000000 & 165.000000 & -180.000000 & 0.284053 & 0.933695 & 178.010860 & 1.000000 & -1.000000 & -0.999397 & 0.983007 & -0.993821 & -0.991761 & -0.997940 \\
C & 3 & 0.000000 & -85.9, 0.0, 85.9 & 0.33, 0.33, 0.33 & 0.000000 & 0.000000 & 0.000000 & 0.000000 & 1.498476 & 0.727474 & 0.000000 & 1.000000 & 1.000000 & 1.000000 & 0.974253 & 0.993306 & 0.988157 & 0.9


## 7. Suggested journal interpretation

Use this result as **hardware-backed geometry validation**, not as the full QFL training evaluation.

The safe claim is:

> IBM hardware validation shows that linear aggregation can produce a geometrically misleading quantum parameter and therefore an incorrect physical observable. The midpoint-projected manifold update provides a server-side geometry mechanism that respects the periodic structure of variational quantum parameters.

The full QFL training experiments should separately compare FedAvg, QoS-only A2G, Euclidean A2G, circular A2G, midpoint-projected A2G, and adaptive midpoint-projected A2G.
